# Software Engineer → AI Researcher: your first real LLM experiment

*(Research Engineer · ML Researcher · LLMs · RL …)*

In ~10 minutes, on a free GPU, you will **train a real language model and run your first architecture experiment** — then get a result you can post.

The way in is simple: **do small experiments, improve every time, post them.** Other researchers give feedback, and you get into the field. This notebook is one full loop.

**Today's experiment:** does swapping the activation function (squared ReLU → SwiGLU) lower the loss at small scale?

Just run each cell top to bottom (`Shift+Enter`).

## Step 0 — Turn on the GPU

Menu: **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. Then run the cell below — it should print a GPU name, not `NO GPU`.

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU ready:', torch.cuda.get_device_name(0))
else:
    print('NO GPU — go to Runtime > Change runtime type > T4 GPU, then re-run this cell.')

## Step 1 — Clone the research kit

This is the open codebase you'll experiment in (modular transformer with GQA, RoPE, RMSNorm, Muon optimizer).

In [ ]:
!git clone --depth 1 https://github.com/vukrosic/llm-research-kit.git
%cd llm-research-kit

## Step 2 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## Step 3 — Download a small dataset (40M tokens)

A subset of the pretraining data — fast to download, enough for a tiny model. (Bigger 1B / 2B sets are in the README for later.)

In [ ]:
from datasets import load_dataset
import os
print('Downloading 40M-token subset...')
ds = load_dataset('vukrosic/blueberry-1B-pretrain', split='train[:20000]')
os.makedirs('processed_data/speedrun_40M', exist_ok=True)
ds.save_to_disk('processed_data/speedrun_40M')
print('Data ready -> processed_data/speedrun_40M')

## Step 4 — Train the BASELINE

A ~1M-parameter transformer on 3M tokens (`--config tiny1m`). This is the **current recipe** — the number you're trying to beat. Watch the `Val Loss` drop as it trains (a few minutes).

In [ ]:
!python train_llm.py --config tiny1m --seed 42 2>&1 | tee baseline.log

## Step 5 — Your FIRST experiment: change ONE thing

The baseline uses a **squared ReLU** activation in the feed-forward layer. You'll swap it for **SwiGLU** (used in LLaMA) and keep *everything else identical* — same seed, same data, same size. That's a clean ablation: one variable changed.

The only difference from Step 4 is `--ffn_variant swiglu`.

In [ ]:
!python train_llm.py --config tiny1m --ffn_variant swiglu --seed 42 2>&1 | tee experiment.log

## Step 6 — Compare: did your change help?

Lower val loss = better. This reads both runs, prints the final numbers, and plots the two loss curves.

In [ ]:
import re
import matplotlib.pyplot as plt

def parse_log(path):
    steps, losses = [], []
    for line in open(path):
        m = re.search(r'Step (\d+): Val Loss: ([\d.]+)', line)
        if m:
            steps.append(int(m.group(1)))
            losses.append(float(m.group(2)))
    return steps, losses

bs, bl = parse_log('baseline.log')
es, el = parse_log('experiment.log')

print(f'Baseline   (squared ReLU) final val loss: {bl[-1]:.4f}')
print(f'Experiment (SwiGLU)       final val loss: {el[-1]:.4f}')
delta = el[-1] - bl[-1]
verdict = 'BETTER (lower loss)' if delta < 0 else 'WORSE (higher loss)'
print(f'Delta: {delta:+.4f}  ->  SwiGLU is {verdict} at this scale')

plt.figure(figsize=(7, 4))
plt.plot(bs, bl, 'o-', label='baseline (squared ReLU)')
plt.plot(es, el, 's-', label='experiment (SwiGLU)')
plt.xlabel('training step'); plt.ylabel('validation loss')
plt.title('My first ablation: activation function')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('my_first_experiment.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved chart -> my_first_experiment.png (attach this to your post)')

## Step 7 — Post it

Posting is how you get into the field. This generates a ready-to-paste update from your actual numbers — drop it on X / LinkedIn with the chart from Step 6.

In [ ]:
better = 'beat' if delta < 0 else 'lost to'
post = f'''Day 1 of switching from software engineer -> AI researcher.

Ran my first real LLM ablation on a single GPU:
- ~1M-param transformer, 3M tokens
- changed one thing: activation squared ReLU -> SwiGLU
- baseline val loss {bl[-1]:.3f} | SwiGLU {el[-1]:.3f} ({delta:+.3f})

SwiGLU {better} the baseline at this scale. Next: test it on a bigger model.

Kit: github.com/vukrosic/llm-research-kit'''
print(post)

## You just did real research. What now?

Change **one** thing and re-run Steps 5–7. Each one is a new experiment and a new post:

| Try | Flag |
|---|---|
| Different activation | `--ffn_variant gelu` (or `satrelu`) |
| LayerNorm instead of RMSNorm | `--use_layernorm true` |
| More / fewer KV heads (GQA) | `--n_kv_heads 4` |
| Sliding-window attention | `--use_sliding_window true` |
| Parallel attn+FFN block (PaLM/GPT-J) | `--use_parallel_block true` |
| Attention sink | `--use_attn_sink true` |

Always change **one** variable, keep `--seed 42`, and compare to the baseline. That's the whole game.

---

### Level up: let an AI agent run experiments for you

Once you outgrow Colab, do it on a bigger remote GPU with an autonomous coding agent:

1. Install **Claude Code** or **Codex**.
2. Rent a remote GPU (you'll set up SSH + a key — the AI can walk you through it). Tutorial: https://youtu.be/FXUMISiMOTE
3. Give the agent your SSH command and tell it to clone the kit: https://github.com/vukrosic/llm-research-kit
4. Tell it to install requirements and download the 1B dataset from the README.
5. Paste this:
   > *"We are doing research. First recommend experiments on different architecture changes — query, key, attention, activation functions, residual connections, softmax, etc. — and talk to me about it."*
6. It works autonomously; just talk to it. If something is complex, tell it to explain the next step in **one sentence**.

Post your experiments daily or weekly. That's how you become a researcher in public.

---

**Stuck or want a faster path?**
- Do AI research with us on Discord: https://discord.gg/6AbXGpKTwN
- 1-on-1: I'll get your first experiment running on your machine, around your goal → https://cal.com/vuk-ai/60-min

**What's the first experiment you have in mind?**